# Food Access Analysis — Washington State

This notebook explores food access and socioeconomic indicators across Washington State census tracts. It draws from three data sources: USDA food access data, ACS 5-year Census estimates, and TIGER/Line census tract shapefiles.

## 1. Setup

### 1.1 Import Libraries

In [1]:
# Importing libraries
import requests
import os
import pandas as pd
import geopandas as gpd

from census import Census
from us import states
from dotenv import load_dotenv

### 1.2 Load Environment Variables & API Key

In [ ]:

# Some os work with a .env file to ensure security of API Key
load_dotenv()

API_KEY = os.getenv("CENSUS_API_KEY")
print(repr(API_KEY))

## 2. Data Ingestion

### 2.1 Census API Helper

The `census` library was abandoned due to consistent errors. Census data is fetched manually via the ACS 5-year API endpoint.

In [19]:
# Going to abandon census
# Errors are consistent with the library

# Manually request the information
def fetch_acs5_tracts(fields, state_fips, key, year = 2024): 
    fields_str = ",".join(["NAME"] + list(fields))
    
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5"
        f"?get={fields_str}"
        f"&for=tract:*"
        f"&in=state:{state_fips}"
        f"&key={key}"
    )
    
    response = requests.get(url)
    data = response.json()
    
    return pd.DataFrame(data[1:], columns = data[0])

### 2.2 Load Raw DataFrames

Three raw sources are loaded: the USDA food access CSV, ACS 5-year Census tract data, and the Washington State census tract shapefile.

In [20]:
# Creating dataframes

# Using the USDA csv file
raw_usda_df = pd.read_csv("../data/raw/StateAndCountyData.csv")

# Using the Census API Key
raw_census = fetch_acs5_tracts(
    fields = (
        "B19013_001E",
        "B17001_002E",
        "B17001_001E",
        "B25044_003E",
        "B25044_010E",
        "B25044_001E",
        "B03002_003E",
        "B03002_004E",
        "B03002_012E",
        "B03002_001E",
        "B22003_002E",
        "B22003_001E",
    ),
    state_fips = "53",
    key = API_KEY, 
)

raw_census_df = pd.DataFrame(raw_census)


# Using the shp file to get geography with geopandas
raw_geometry_gdf = gpd.read_file("../data/raw/tl_2025_53_tract.shp")

In [35]:
# Variables List to be merged with raw_usda_df
raw_usda_varaibles_df = pd.read_csv(r"..\data\raw\VariableList.csv")


### 2.3 API Connectivity Debug

Sanity check that the Census API endpoint responds correctly before proceeding.

In [7]:
# Running into an issue
# Going to check ip throttle
# Using requests library

url = f"https://api.census.gov/data/2024/acs/acs5?get=NAME,B19013_001E,B25044_003E,B17001_002E,B17001_001E,B25044_010E,B25044_001E,B03002_003E,B03002_004E,B03002_012E,B03002_001E,B22003_002E,B22003_001E&for=tract:*&in=state:53&in=county:*&key={API_KEY}"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

# Request is working... 

200
[["NAME","B19013_001E","B25044_003E","B17001_002E","B17001_001E","B25044_010E","B25044_001E","B03002_003E","B03002_004E","B03002_012E","B03002_001E","B22003_002E","B22003_001E","state","county","tract"],
["Census Tract 9501; Adams County; Washington","68750","6","314","2681","57","1111","2281","16","144","2762","120","1111","53","001","950100"],
["Census Tract 9502; Adams County; Washington","72500","14","267","2041","0","610","1199","17","696","2041","59","610","53","001","950200"],
["Census Tr


# Checks for each of the 3 raw dataframes to ensure data integrity


In [22]:
# USDA Data
print(raw_usda_df.head())
print("=========================================================================================================")

# Describe
print(raw_usda_df.describe())
print("=========================================================================================================")

# Info
print(raw_usda_df.info())

     FIPS State   County          Variable_Code        Value
0  1001.0    AL  Autauga          LACCESS_POP15  18092.66135
1  1001.0    AL  Autauga          LACCESS_POP19  18503.22551
2  1001.0    AL  Autauga  PCH_LACCESS_POP_15_19      2.26923
3  1001.0    AL  Autauga      PCT_LACCESS_POP15     33.15435
4  1001.0    AL  Autauga      PCT_LACCESS_POP19     33.90670
                FIPS         Value
count  957278.000000  9.577530e+05
mean    30343.740874  1.114661e+04
std     15183.939474  6.302679e+05
min      1001.000000 -9.999000e+03
25%     18169.000000  0.000000e+00
50%     29171.000000  2.738841e+00
75%     45079.000000  3.058333e+01
max     56045.000000  3.380466e+08
<class 'pandas.DataFrame'>
RangeIndex: 957753 entries, 0 to 957752
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   FIPS           957278 non-null  float64
 1   State          957753 non-null  str    
 2   County         957753 non-null  str

In [23]:
# Census Data
print(raw_census_df.head())
print("=========================================================================================================")

# Describe
print(raw_census_df.describe())
print("=========================================================================================================")

# Info
print(raw_census_df.info())

                                             NAME B19013_001E B17001_002E  \
0     Census Tract 9501; Adams County; Washington       68750         314   
1     Census Tract 9502; Adams County; Washington       72500         267   
2  Census Tract 9503.01; Adams County; Washington       58585         117   
3  Census Tract 9503.02; Adams County; Washington       65893         913   
4  Census Tract 9503.03; Adams County; Washington       75673         315   

  B17001_001E B25044_003E B25044_010E B25044_001E B03002_003E B03002_004E  \
0        2681           6          57        1111        2281          16   
1        2041          14           0         610        1199          17   
2        1762           0           0         593         576           0   
3        2981          15          35         779         264           0   
4        2384           0           0         697         578           0   

  B03002_012E B03002_001E B22003_002E B22003_001E state county   tract  
0

In [24]:
# USDA Data
print(raw_geometry_gdf.head())
print("=========================================================================================================")

# Describe
print(raw_geometry_gdf.describe())
print("=========================================================================================================")

# Info
print(raw_geometry_gdf.info())

  STATEFP COUNTYFP TRACTCE        GEOID               GEOIDFQ     NAME  \
0      53      071  920301  53071920301  1400000US53071920301  9203.01   
1      53      071  920302  53071920302  1400000US53071920302  9203.02   
2      53      071  920901  53071920901  1400000US53071920901  9209.01   
3      53      033  030003  53033030003  1400000US53033030003   300.03   
4      53      033  005307  53033005307  1400000US53033005307    53.07   

               NAMELSAD  MTFCC FUNCSTAT     ALAND  AWATER     INTPTLAT  \
0  Census Tract 9203.01  G5020        S   2621059       0  +46.0422556   
1  Census Tract 9203.02  G5020        S   2919497       0  +46.0505214   
2  Census Tract 9209.01  G5020        S  21809260  189083  +46.0689693   
3   Census Tract 300.03  G5020        S   4604664  963078  +47.3588458   
4    Census Tract 53.07  G5020        S    130030       0  +47.6630600   

       INTPTLON                                           geometry  
0  -118.3749820  POLYGON ((-118.38863 46.

In [43]:
# VariablesList Data
print(raw_usda_varaibles_df.head())
print("========================================================")
print()

print(raw_usda_varaibles_df["Variable_Name"].unique())
print("========================================================")


                                       Variable_Name  \
0              Population, low access to store, 2015   
1              Population, low access to store, 2019   
2  Population, low access to store (% change), 20...   
3          Population, low access to store (%), 2015   
4          Population, low access to store (%), 2019   

                       Category_Name Category_Code Subcategory_Name  \
0  Access and Proximity to Foodstore        ACCESS          Overall   
1  Access and Proximity to Foodstore        ACCESS          Overall   
2  Access and Proximity to Foodstore        ACCESS          Overall   
3  Access and Proximity to Foodstore        ACCESS          Overall   
4  Access and Proximity to Foodstore        ACCESS          Overall   

           Variable_Code     Units  
0          LACCESS_POP15     Count  
1          LACCESS_POP19     Count  
2  PCH_LACCESS_POP_15_19  % change  
3      PCT_LACCESS_POP15   Percent  
4      PCT_LACCESS_POP19   Percent  

<ArrowStringA

### 3.5 Null Value Audit

In [27]:
# Information seems to be in order. OTher things we can check are null values and missing values

print("Number of null values within Census DataFrame")
print(raw_census_df.isna().any().sum())
print()

print("Number of null values within USDA DataFrame")
print(raw_usda_df.isna().any().sum())
print()

print("Number of null values within Geometry DataFrame")
print(raw_geometry_gdf.isna().any().sum())

Number of null values within Census DataFrame
0

Number of null values within USDA DataFrame
1

Number of null values within Geometry DataFrame
0


# Doing Some Data Conversions

### 4.1 Inspect USDA Column Schema

In [44]:
print(raw_usda_df.columns)

Index(['FIPS', 'State', 'County', 'Variable_Code', 'Value'], dtype='str')


### 4.2 Filter USDA Data to Washington State & Merge Variable Labels

In [47]:
# The USDA csv file needs to be isolated to just Washington state. Then it needs to be isolated further
print(raw_usda_df[raw_usda_df["State"] == "WA"].count())

washington_usda_df = raw_usda_df[raw_usda_df["State"] == "WA"]
# Isolated from ~96k + rows down to ~11k

# Now we need to combine the washington dataset with the given labels
combined_usda_df = pd.merge(
    washington_usda_df, 
    raw_usda_varaibles_df, 
    on = "Variable_Code", 
    how = "left"
)

FIPS             11856
State            11856
County           11856
Variable_Code    11856
Value            11856
dtype: int64


In [54]:
print(combined_usda_df.columns)
print("========================================================")
print(combined_usda_df["Category_Code"].unique())
print("========================================================")
print(combined_usda_df["Category_Name"].unique())

Index(['FIPS', 'State', 'County', 'Variable_Code', 'Value', 'Variable_Name',
       'Category_Name', 'Category_Code', 'Subcategory_Name', 'Units'],
      dtype='str')
<ArrowStringArray>
[       'ACCESS',    'ASSISTANCE',        'HEALTH',    'INSECURITY',
         'LOCAL',   'RESTAURANTS', 'SOCIOECONOMIC',        'STORES',
  'PRICES_TAXES']
Length: 9, dtype: str
<ArrowStringArray>
['Access and Proximity to Foodstore',                   'Food Assistance',
      'Health and Physical Activity',             'State Food Insecurity',
                       'Local Foods',          'Restaurant Availability ',
           'Restaurant Availability',     'Socioeconomic Characteristics',
                'Store Availability',                        'Food Taxes']
Length: 10, dtype: str


# Adjusting Census Data

### 5.1 Rename Census Columns to Descriptive Names

In [ ]:
# Should have done this earlier. Adjusting the column names to something more meaningful.

column_names = {
    "B19013_001E": "median_income",
    "B17001_002E": "poverty_population",
    "B17001_001E": "poverty_total", # 
    "B25044_003E": "no_vehicle_owner", # No vehicle period
    "B25044_010E": "no_vehicle_renter", # No vehicle, but rents a vehicle
    "B25044_001E": "vehicle_total",
    "B03002_003E": "pop_white",
    "B03002_004E": "pop_black",
    "B03002_012E": "pop_hispanic",
    "B03002_001E": "pop_total",
    "B22003_002E": "snap_households",
    "B22003_001E": "snap_total",
    "NAME":        "tract_name",
    "state":       "state_fips",
    "county":      "county_fips",
    "tract":       "tract_code"
}

raw_census_df = raw_census_df.rename(columns = column_names)

### 5.2 Initialize Feature-Engineered DataFrame & Build GEOID

In [91]:
print(raw_census_df)

# Creating a new dataframe to house features after feature engineering
census_df = pd.DataFrame()

census_df['GEOID'] = raw_census_df['state'] + raw_census_df['county'] + raw_census_df['tract']

                                           tract_name  median_income  \
0         Census Tract 9501; Adams County; Washington          68750   
1         Census Tract 9502; Adams County; Washington          72500   
2      Census Tract 9503.01; Adams County; Washington          58585   
3      Census Tract 9503.02; Adams County; Washington          65893   
4      Census Tract 9503.03; Adams County; Washington          75673   
...                                               ...            ...   
1779  Census Tract 9400.03; Yakima County; Washington          66173   
1780  Census Tract 9400.05; Yakima County; Washington          71607   
1781  Census Tract 9400.06; Yakima County; Washington          63472   
1782  Census Tract 9400.07; Yakima County; Washington          52076   
1783  Census Tract 9400.08; Yakima County; Washington          56000   

      poverty_population  poverty_total  no_vehicle_owner  no_vehicle_renter  \
0                    314           2681                

KeyError: 'state'

### 5.3 Identify & Convert Numeric Columns

In [92]:
# Function will be defined to make pd.to_numeric easier
def get_numeric_columns(dataframe): 
    # Grabs all columns into a temporary dataframe
    df_cols = dataframe.select_dtypes(include = ['str']).columns
    # Defines where the convertible columns will go
    convertible = []
    
    # For each column
    for column in df_cols:
        # Try to convert the column to a numeric value, then add that column to the list 
        try: 
            pd.to_numeric(dataframe[column], errors = 'raise')
            convertible.append(column)
        # Otherwise, raise and catch an error and skip past this iteration
        except (ValueError, TypeError): 
            continue
        
    return convertible

In [93]:
# Checking which dataframe values can be converted to numeric values if necessary
convertible_columns = get_numeric_columns(raw_census_df)
print(convertible_columns)

for col in convertible_columns: 
    raw_census_df[col] = pd.to_numeric(raw_census_df[col], errors = 'coerce')
    
print(raw_census_df.info())


[]
<class 'pandas.DataFrame'>
RangeIndex: 1784 entries, 0 to 1783
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   tract_name          1784 non-null   str  
 1   median_income       1784 non-null   int64
 2   poverty_population  1784 non-null   int64
 3   poverty_total       1784 non-null   int64
 4   no_vehicle_owner    1784 non-null   int64
 5   no_vehicle_renter   1784 non-null   int64
 6   vehicle_total       1784 non-null   int64
 7   pop_white           1784 non-null   int64
 8   pop_black           1784 non-null   int64
 9   pop_hispanic        1784 non-null   int64
 10  pop_total           1784 non-null   int64
 11  snap_households     1784 non-null   int64
 12  snap_total          1784 non-null   int64
 13  state_fips          1784 non-null   int64
 14  county_fips         1784 non-null   int64
 15  tract_code          1784 non-null   int64
dtypes: int64(15), str(1)
memory usage: 301.7 KB
None


### 5.4 Feature Engineering — Derived Statistics

In [94]:
# Columns are now converted. Build out statistics
census_df['median_income'] = raw_census_df['median_income']
census_df['poverty_percentage'] = raw_census_df['poverty_population'] / raw_census_df['poverty_total']
census_df['no_vehicle_percentage'] = (raw_census_df['no_vehicle_owner'] + raw_census_df['no_vehicle_renter']) / raw_census_df['vehicle_total']
census_df['snap_percentage'] = raw_census_df['snap_households'] / raw_census_df['snap_total']
census_df['percent_black'] = raw_census_df['pop_black'] / raw_census_df['pop_total']
census_df['percent_hispanic'] = raw_census_df['pop_hispanic'] / raw_census_df['pop_total']
census_df['percent_white'] = raw_census_df['pop_white'] / raw_census_df['pop_total']

census_df = census_df.sort_values(by=['median_income', 'poverty_percentage'], ascending = False)

In [95]:
print(census_df.head())

     median_income  poverty_percentage  no_vehicle_percentage  \
634         250001            0.082272               0.032512   
822         250001            0.053816               0.000463   
450         250001            0.053769               0.034111   
414         250001            0.053269               0.042678   
621         250001            0.046790               0.056436   

     snap_percentage  percent_black  percent_hispanic  percent_white  
634         0.008867       0.036782          0.000000       0.222757  
822         0.000463       0.000000          0.054917       0.504617  
450         0.007181       0.055330          0.013356       0.745290  
414         0.024283       0.002751          0.057264       0.815954  
621         0.015842       0.000000          0.003074       0.579918  


### 5.5 Outlier Detection & Cleaning

In [96]:
#Check for outliers
print(census_df.isna().any().sum())

census_df.dropna(inplace = True)
print(census_df.isna().any().sum())

print(census_df.sort_values(by="median_income"))


6
0
      median_income  poverty_percentage  no_vehicle_percentage  \
1161     -666666666            0.308642               0.000000   
1259     -666666666            0.000000               0.230769   
538      -666666666            0.302303               0.181247   
435           17500            0.472441               0.806452   
1470          18929            0.493222               0.532597   
...             ...                 ...                    ...   
414          250001            0.053269               0.042678   
450          250001            0.053769               0.034111   
822          250001            0.053816               0.000463   
634          250001            0.082272               0.032512   
816          250001            0.010469               0.000000   

      snap_percentage  percent_black  percent_hispanic  percent_white  
1161         0.000000       0.150935          0.229742       0.470169  
1259         0.000000       0.000000          0.000000     

In [89]:
# Some other outliers that are going to mess with the data
census_df = census_df[census_df['median_income'] >= 0]
print(census_df.sort_values(by = 'median_income'))

      median_income  poverty_percentage  no_vehicle_percentage  \
435           17500            0.472441               0.806452   
1470          18929            0.493222               0.532597   
1722          21457            0.522690               0.081930   
1721          21970            0.578024               0.075643   
1035          23457            0.383484               0.461385   
...             ...                 ...                    ...   
1363         250001            0.034263               0.019024   
831          250001            0.034980               0.000000   
822          250001            0.053816               0.000463   
634          250001            0.082272               0.032512   
835          250001            0.010964               0.024136   

      snap_percentage  percent_black  percent_hispanic  percent_white  
435          0.000000       0.047299          0.073285       0.372263  
1470         0.561514       0.054897          0.084039       0.